In [ ]:
!pip install geopandas folium requests shapely pandas -q

In [ ]:
import os, json, time, requests, pickle
import pandas as pd
import geopandas as gp
from shapely.geometry import shape, mapping, Point
from shapely.ops import unary_union

MAPBOX_TOKEN = "hidden for privacy"

## Data Loading and Cleaning

In [ ]:
df = pd.read_csv("gym_track_new.csv")

def assign_category(row):
    price = str(row["price"]).strip()
    types = str(row["types"]).lower() if pd.notna(row["types"]) else ""
    if price == "outdoor track/gym": return "Outdoor track / gym"
    if price == "community center": return "Community center"
    if price == "University": return "University facility"
    if "yoga_studio" in types: return "Yoga / Pilates"
    return "Gym / fitness center"

def assign_price_tier(price):
    price = str(price).strip()
    if price == "outdoor track/gym": return "Free (outdoor)"
    if price == "community center": return "Community center"
    if price == "University": return "University"
    if price == "personal": return "Personal training"
    if price in ["N/A", "nan", ""]: return "Unknown"
    try:
        val = float(price.replace("$", "").replace(",", ""))
        if val <= 30: return "Budget ($0-30)"
        if val <= 80: return "Mid-range ($31-80)"
        if val <= 150: return "Premium ($81-150)"
        return "Luxury ($150+)"
    except: return "Unknown"

df["category"] = df.apply(assign_category, axis=1)
df["price_tier"] = df["price"].apply(assign_price_tier)
df = df.dropna(subset=["latitude", "longitude"])

gyms = gp.GeoDataFrame(
    df, geometry=gp.points_from_xy(df["longitude"], df["latitude"]), crs="EPSG:4326"
)

## ZIP Code Boundary using ZCTA shapefile

In [ ]:
TARGET_ZIPS = [
    "02108","02109","02110","02111","02113","02114","02115","02116",
    "02118","02119","02120","02121","02122","02124","02125","02127",
    "02128","02129","02130","02133","02134","02135","02138","02139",
    "02141","02142","02163","02199","02203","02205","02210","02215",
    "02445","02446","02458","02459","02467","02472",
]

zcta_all = gp.read_file("cb_2020_us_zcta520_500k.shp")
target_zcta = zcta_all[zcta_all["ZCTA5CE20"].isin(TARGET_ZIPS)]
study_area = unary_union(target_zcta.geometry)

## MBTA lines

In [ ]:
import requests, json

route_list = [
    ("Red", "#DA291C"),
    ("Orange", "#ED8B00"),
    ("Blue", "#003DA5"),
    ("Green-B", "#00843D"),
    ("Green-C", "#00843D"),
    ("Green-D", "#00843D"),
    ("Green-E", "#00843D"),
]

def decode_polyline(polyline):
    coords = []
    index = 0
    lat = 0
    lng = 0
    while index < len(polyline):
        shift = 0
        result = 0
        while True:
            b = ord(polyline[index]) - 63
            index += 1
            result |= (b & 0x1f) << shift
            shift += 5
            if b < 0x20:
                break
        lat += (~(result >> 1) if (result & 1) else (result >> 1))
        shift = 0
        result = 0
        while True:
            b = ord(polyline[index]) - 63
            index += 1
            result |= (b & 0x1f) << shift
            shift += 5
            if b < 0x20:
                break
        lng += (~(result >> 1) if (result & 1) else (result >> 1))
        coords.append([lng / 1e5, lat / 1e5])
    return coords

line_features = []
for route_id, color in route_list:
    url = f"https://api-v3.mbta.com/shapes?filter[route]={route_id}"
    resp = requests.get(url)
    data = resp.json()
    for shape in data.get("data", []):
        polyline = shape["attributes"]["polyline"]
        coords = decode_polyline(polyline)
        line_features.append({
            "type": "Feature",
            "properties": {"color": color, "route": route_id},
            "geometry": {"type": "LineString", "coordinates": coords}
        })

mbta_lines = {"type": "FeatureCollection", "features": line_features}

stops_url = "https://api-v3.mbta.com/stops?filter[route]=Red,Orange,Blue,Green-B,Green-C,Green-D,Green-E"
stops_resp = requests.get(stops_url)
stops_data = stops_resp.json()

stop_features = []
for stop in stops_data["data"]:
    attrs = stop["attributes"]
    if attrs["latitude"] and attrs["longitude"]:
        stop_features.append({
            "type": "Feature",
            "properties": {"STATION": attrs["name"]},
            "geometry": {"type": "Point", "coordinates": [attrs["longitude"], attrs["latitude"]]}
        })

mbta_stations = {"type": "FeatureCollection", "features": stop_features}

## Get walking isorchrones from MapBox (cache if needed)

In [ ]:
def get_isochrone(lon, lat, minutes=[10, 20, 30], profile="walking"):
    url = (
        f"https://api.mapbox.com/isochrone/v1/mapbox/{profile}/"
        f"{lon},{lat}"
        f"?contours_minutes={','.join(str(m) for m in minutes)}"
        f"&polygons=true&denoise=1"
        f"&access_token={MAPBOX_TOKEN}"
    )
    resp = requests.get(url)
    if resp.status_code == 200:
        return resp.json()
    return None

time_bands = [10, 20, 30]
walk_cache = "gym_isochrones_walk_v4.pkl"

if os.path.exists(walk_cache):
    with open(walk_cache, "rb") as f:
        walk_isochrones = pickle.load(f)
else:
    walk_isochrones = []
    for i, (_, gym) in enumerate(gyms.iterrows()):
        lon, lat = gym["longitude"], gym["latitude"]
        name = gym["name"]
        result = get_isochrone(lon, lat, minutes=time_bands, profile="walking")
        if result and "features" in result:
            for feature in result["features"]:
                mins = feature["properties"]["contour"]
                geom = shape(feature["geometry"])
                if geom.is_valid:
                    clipped = geom.intersection(study_area)
                    if not clipped.is_empty:
                        walk_isochrones.append({
                            "name": name,
                            "category": gym["category"],
                            "price_tier": gym["price_tier"],
                            "band": mins,
                            "geometry": clipped,
                        })
            print("SUCCESS")
        else:
            print("FAILED")
        if i < len(gyms) - 1:
            time.sleep(0.25)
    with open(walk_cache, "wb") as f:
        pickle.dump(walk_isochrones, f)

## Pre-merge isochrones for filter combos

In [ ]:
categories = sorted(gyms["category"].unique())
price_tiers = sorted(gyms["price_tier"].unique())

def merge_isochrones(iso_list):
    data = {}
    for cat in categories:
        for pt in price_tiers:
            for band in time_bands:
                m = [iso["geometry"] for iso in iso_list
                     if iso["category"] == cat and iso["price_tier"] == pt and iso["band"] == band]
                if m: data[f"{cat}|{pt}|{band}"] = mapping(unary_union(m))
    for pt in price_tiers:
        for band in time_bands:
            m = [iso["geometry"] for iso in iso_list if iso["price_tier"] == pt and iso["band"] == band]
            if m: data[f"All|{pt}|{band}"] = mapping(unary_union(m))
    for cat in categories:
        for band in time_bands:
            m = [iso["geometry"] for iso in iso_list if iso["category"] == cat and iso["band"] == band]
            if m: data[f"{cat}|All|{band}"] = mapping(unary_union(m))
    for band in time_bands:
        m = [iso["geometry"] for iso in iso_list if iso["band"] == band]
        if m: data[f"All|All|{band}"] = mapping(unary_union(m))
    return data

walk_data = merge_isochrones(walk_isochrones)

gym_markers_json = []
for _, gym in gyms.iterrows():
    gym_markers_json.append({
        "name": str(gym["name"]),
        "lat": float(gym["latitude"]),
        "lon": float(gym["longitude"]),
        "category": str(gym["category"]),
        "price_tier": str(gym["price_tier"]),
        "price": str(gym["price"]),
        "address": str(gym.get("address", "")),
    })

boundary_geojson = mapping(study_area)
cat_price_map = {}
for cat in categories:
    valid_prices = sorted(gyms[gyms["category"] == cat]["price_tier"].unique().tolist())
    cat_price_map[cat] = valid_prices
cat_price_map["All"] = sorted(gyms["price_tier"].unique().tolist())

## Accessibility score grid

In [ ]:
import numpy as np
from shapely.geometry import Point, box, mapping
from shapely.prepared import prep
from shapely.strtree import STRtree

price_rank = {
    'Free (outdoor)': 0, 
    'Budget ($0-30)': 1, 
    'Community center': 1,
    'Mid-range ($31-80)': 2, 
    'Premium ($81-150)': 3,
    'Luxury ($150+)': 4, 
    'University': 1, 
    'Personal training': 3, 
    'Unknown': 3
}
band_rank = {10: 0, 20: 1, 30: 2}
MAX_SCORE = 6

CELL_SIZE = 0.0009  # this was the smallest I could do without the page taking too long to load

bounds = study_area.bounds
xs = np.arange(bounds[0], bounds[2], CELL_SIZE)
ys = np.arange(bounds[1], bounds[3], CELL_SIZE)

iso_index = []
iso_geoms = []
for iso in walk_isochrones:
    iso_index.append({
        'band': iso['band'],
        'price_tier': iso['price_tier'],
        'category': iso['category'],
    })
    iso_geoms.append(iso['geometry'])

tree = STRtree(iso_geoms)
study_prep = prep(study_area)

score_grid_all = []
score_grid_by_cat = {}

for cat in categories:
    score_grid_by_cat[cat] = []

total = len(xs) * len(ys)
done = 0
for yi, y in enumerate(ys):
    if yi % 10 == 0:
        print(f"  Row {yi}/{len(ys)} ({done}/{total} cells)...", flush=True)
    for xi, x in enumerate(xs):
        done += 1
        cx = x + CELL_SIZE / 2
        cy = y + CELL_SIZE / 2
        pt = Point(cx, cy)
        if not study_prep.contains(pt):
            continue
        candidates = tree.query(pt)
        best_all = None
        best_by_cat = {}
        for idx in candidates:
            iso = iso_index[idx]
            geom = iso_geoms[idx]
            if geom.contains(pt):
                s = band_rank[iso['band']] + price_rank.get(iso['price_tier'], 3)
                cat = iso['category']
                if best_all is None or s < best_all:
                    best_all = s
                if cat not in best_by_cat or s < best_by_cat[cat]:
                    best_by_cat[cat] = s
        if best_all is None and not best_by_cat:
            continue
        cell_box = box(x, y, x + CELL_SIZE, y + CELL_SIZE)
        clipped = cell_box.intersection(study_area)
        if clipped.is_empty:
            continue
        clipped_geojson = mapping(clipped)
        if best_all is not None:
            score_grid_all.append({"s": best_all, "g": clipped_geojson})
        for cat, s in best_by_cat.items():
            score_grid_by_cat[cat].append({"s": s, "g": clipped_geojson})

score_data = {"All": score_grid_all}
for cat in categories:
    score_data[cat] = score_grid_by_cat[cat]

score_data_json = json.dumps(score_data)

## HTML map builder

In [ ]:
cat_options = '\n'.join(f'      <option value="{c}">{c}</option>' for c in categories)
price_options = '\n'.join(f'      <option value="{p}">{p}</option>' for p in price_tiers)

html_template = """<!DOCTYPE html>
<html><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Boston gym accessibility — filterable map</title>
<link href="https://fonts.googleapis.com/css2?family=Nunito:wght@400;500;700&display=swap" rel="stylesheet">
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css"/>
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>
<style>
  * { margin:0; padding:0; box-sizing:border-box; }
  body { font-family: "Nunito", sans-serif; }
  #map { position:absolute; top:0; left:0; right:0; bottom:0; }
  .controls { position:absolute; top:12px; left:60px; z-index:1000; background:white;
    padding:14px 18px; border-radius:10px; box-shadow:0 2px 12px rgba(0,0,0,0.12);
    font-size:13px; max-width:320px; }
  .controls h3 { font-size:14px; font-weight:500; margin-bottom:10px; }
  .filter-row { margin-bottom:8px; }
  .filter-row label { display:block; font-size:11px; color:#888; margin-bottom:3px; }
  .filter-row select { width:100%; padding:6px 8px; border:1px solid #d3d1c7; border-radius:6px;
    font-size:13px; background:white; }
  .check-row { margin-bottom:8px; font-size:12px; display:flex; align-items:center; gap:6px; }
  .check-row input { margin:0; }
  .stats { margin-top:10px; padding-top:8px; border-top:1px solid #e8e7e2; font-size:11px; color:#888; }
  .legend { position:absolute; bottom:30px; left:12px; z-index:1000; background:white;
    padding:12px 16px; border-radius:8px; box-shadow:0 2px 8px rgba(0,0,0,0.1); font-size:12px; }
  .legend-title { font-weight:500; margin-bottom:6px; }
  .legend-item { display:flex; align-items:center; gap:6px; margin:3px 0; }
  .legend-swatch { width:14px; height:14px; border-radius:3px; display:inline-block; }
  .legend-line { width:20px; height:3px; display:inline-block; border-radius:1px; }
  .legend-section { margin-top:8px; padding-top:6px; border-top:1px solid #e8e7e2; }
  .legend-dot { width:10px; height:10px; border-radius:50%; display:inline-block; }
  .leaflet-popup-content { font-family:"Nunito",sans-serif; font-size:13px; line-height:1.5; }
  .popup-name { font-weight:600; font-size:14px; margin-bottom:4px; }
  .popup-cat { font-size:11px; padding:2px 8px; border-radius:10px; display:inline-block; margin-bottom:4px; }
  .popup-detail { font-size:12px; color:#666; }
  .mode-btn { flex:1; text-align:center; padding:6px 0; cursor:pointer; transition:all 0.2s; user-select:none; }
  .mode-btn.active { background:#2c2c2a; color:white; }
  .mode-btn:not(.active) { background:white; color:#2c2c2a; }
</style>
</head><body>
<div id="map"></div>
<div class="controls">
  <h3>Filter facilities</h3>
  <div class="filter-row">
    <label>Facility type</label>
    <select id="catFilter" onchange="onCatChange()">
      <option value="All">All types</option>
      %%CAT_OPTIONS%%
    </select>
  </div>
  <div class="filter-row">
    <label>Price tier</label>
    <select id="priceFilter" onchange="updateMap()">
      <option value="All">All prices</option>
      %%PRICE_OPTIONS%%
    </select>
  </div>
  <div class="filter-row" style="margin-top:10px">
    <label>View isochrones by</label>
    <div style="display:flex; border:1px solid #d3d1c7; border-radius:6px; overflow:hidden; font-size:12px;">
      <div class="mode-btn active" onclick="setMode('distance')" id="modeDistance">Distance</div>
      <div class="mode-btn" onclick="setMode('score')" id="modeScore">Score</div>
    </div>
  </div>
  <div class="check-row">
    <input type="checkbox" id="mbtaToggle" onchange="toggleMBTA()">
    <label for="mbtaToggle">Show MBTA lines</label>
  </div>
  <div class="stats" id="stats">Showing all %%TOTAL%% locations</div>
</div>
<div class="legend">
  <div id="legendContent">
    <div class="legend-title">Walking time to nearest facility</div>
    <div class="legend-item"><span class="legend-swatch" style="background:#A300A3"></span> Within 10 min</div>
    <div class="legend-item"><span class="legend-swatch" style="background:#FF00FF"></span> 10&ndash;20 min</div>
    <div class="legend-item"><span class="legend-swatch" style="background:#FF8AFF"></span> 20&ndash;30 min</div>
  </div>
  <div class="legend-section">
    <div class="legend-title">MBTA Rapid Transit</div>
    <div class="legend-item"><span class="legend-line" style="background:#DA291C"></span> Red Line</div>
    <div class="legend-item"><span class="legend-line" style="background:#ED8B00"></span> Orange Line</div>
    <div class="legend-item"><span class="legend-line" style="background:#003DA5"></span> Blue Line</div>
    <div class="legend-item"><span class="legend-line" style="background:#00843D"></span> Green Line</div>
  </div>
</div>
<script>
var map = L.map('map').setView([42.35, -71.08], 13);
L.tileLayer('https://{s}.basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}@2x.png', {
  attribution: 'CartoDB', maxZoom: 19
}).addTo(map);

var gyms = %%GYMS%%;
var walkData = %%WALK_DATA%%;
var boundary = %%BOUNDARY%%;
var mbtaLines = %%MBTA_LINES%%;
var mbtaStations = %%MBTA_STATIONS%%;
var catPriceMap = %%CAT_PRICE_MAP%%;
var scoreData = %%SCORE_DATA%%;
var MAX_SCORE = 6;

var bands = [30, 20, 10];
var colors = {10:'#A300A3', 20:'#FF00FF', 30:'#FF8AFF'};

function scoreColor(s) {
  // 0 (best) -> green, 6 (worst) -> red
  var t = Math.min(s / MAX_SCORE, 1);
  var r, g, b;
  if (t < 0.5) {
    var u = t * 2;
    r = Math.round(39 + u * (253 - 39));
    g = Math.round(174 + u * (216 - 174));
    b = Math.round(96 - u * 61);
  } else {
    var u = (t - 0.5) * 2;
    r = Math.round(253 - u * 24);
    g = Math.round(216 - u * 170);
    b = Math.round(35 + u * 18);
  }
  return 'rgb(' + r + ',' + g + ',' + b + ')';
}

var catColors = {
  'Gym / fitness center': '#378ADD',
  'Yoga / Pilates': '#D4537E',
  'Outdoor track / gym': '#1D9E75',
  'Community center': '#EF9F27',
  'University facility': '#534AB7',
  'Personal training': '#888780'
};

var catBgColors = {
  'Gym / fitness center': '#E6F1FB',
  'Yoga / Pilates': '#FBEAF0',
  'Outdoor track / gym': '#E1F5EE',
  'Community center': '#FAEEDA',
  'University facility': '#EEEDFE',
  'Personal training': '#F1EFE8'
};

L.geoJSON(boundary, {
  style: { fillColor:'transparent', color:'#2c2c2a', weight:2, fillOpacity:0, dashArray:'5,5' }
}).addTo(map);

var mbtaLineGroup = L.layerGroup();
var mbtaStationGroup = L.layerGroup();

L.geoJSON(mbtaLines, {
  style: function(feature) {
    return { color: feature.properties.color || '#888', weight: 3, opacity: 0.8 };
  }
}).addTo(mbtaLineGroup);

L.geoJSON(mbtaStations, {
  pointToLayer: function(feature, latlng) {
    return L.circleMarker(latlng, {
      radius: 4, color: 'white', fillColor: '#2c2c2a',
      fillOpacity: 0.9, weight: 2
    });
  },
  onEachFeature: function(feature, layer) {
    var name = feature.properties.STATION || feature.properties.STOP_NAME || '';
    if (name) layer.bindTooltip(name, {direction:'top', offset:[0,-6]});
  }
}).addTo(mbtaStationGroup);

var markerGroup = L.layerGroup().addTo(map);
var isoGroup = L.layerGroup().addTo(map);
var currentMode = 'distance';

function setMode(mode) {
  currentMode = mode;
  ['Distance','Score'].forEach(function(m) {
    var el = document.getElementById('mode' + m);
    if (m.toLowerCase() === mode) { el.classList.add('active'); }
    else { el.classList.remove('active'); }
  });
  updateLegend();
  updateMap();
}

function updateLegend() {
  var el = document.getElementById('legendContent');
  if (currentMode === 'distance') {
    el.innerHTML = '<div class="legend-title">Walking time to nearest facility</div>' +
      '<div class="legend-item"><span class="legend-swatch" style="background:#A300A3"></span> Within 10 min</div>' +
      '<div class="legend-item"><span class="legend-swatch" style="background:#FF00FF"></span> 10&ndash;20 min</div>' +
      '<div class="legend-item"><span class="legend-swatch" style="background:#FF8AFF"></span> 20&ndash;30 min</div>';
  } else {
    el.innerHTML = '<div class="legend-title">Accessibility score</div>' +
      '<div style="font-size:11px;color:#666;margin-bottom:6px">Combines walking distance + cost</div>' +
      '<div style="display:flex;align-items:center;gap:6px;margin:4px 0">' +
        '<span style="font-size:11px;white-space:nowrap">Best (0)</span>' +
        '<div style="flex:1;height:14px;border-radius:3px;background:linear-gradient(to right,' +
          scoreColor(0) + ',' + scoreColor(1) + ',' + scoreColor(2) + ',' +
          scoreColor(3) + ',' + scoreColor(4) + ',' + scoreColor(5) + ',' + scoreColor(6) + ')"></div>' +
        '<span style="font-size:11px;white-space:nowrap">Worst (6)</span>' +
      '</div>' +
      '<div style="font-size:10px;color:#999;margin-top:4px">0 = free gym within 10 min walk<br>6 = luxury gym, 30 min walk only</div>';
  }
}

function toggleMBTA() {
  var show = document.getElementById('mbtaToggle').checked;
  if (show) { map.addLayer(mbtaLineGroup); map.addLayer(mbtaStationGroup); }
  else { map.removeLayer(mbtaLineGroup); map.removeLayer(mbtaStationGroup); }
}

function onCatChange() {
  var cat = document.getElementById('catFilter').value;
  var priceSelect = document.getElementById('priceFilter');
  var currentPrice = priceSelect.value;

  var validPrices = catPriceMap[cat] || catPriceMap['All'];

  var html = '<option value="All">All prices</option>';
  validPrices.forEach(function(p) {
    html += '<option value="' + p + '"' + (p === currentPrice ? ' selected' : '') + '>' + p + '</option>';
  });
  priceSelect.innerHTML = html;

  if (currentPrice !== 'All' && validPrices.indexOf(currentPrice) === -1) {
    priceSelect.value = 'All';
  }

  updateMap();
}

function makePopup(g) {
  var color = catColors[g.category] || '#888';
  var bg = catBgColors[g.category] || '#f0f0f0';
  var html = '<div class="popup-name">' + g.name + '</div>';
  html += '<span class="popup-cat" style="background:' + bg + ';color:' + color + '">' + g.category + '</span>';
  if (g.address) html += '<div class="popup-detail">' + g.address + '</div>';
  html += '<div class="popup-detail">Price: ' + g.price_tier + '</div>';
  return html;
}

function updateMap() {
  var cat = document.getElementById('catFilter').value;
  var price = document.getElementById('priceFilter').value;

  markerGroup.clearLayers();
  isoGroup.clearLayers();

  var count = 0;
  var matchedGyms = [];
  gyms.forEach(function(g) {
    var matchCat = (cat === 'All' || g.category === cat);
    var matchPrice = (price === 'All' || g.price_tier === price);
    if (matchCat && matchPrice) {
      count++;
      matchedGyms.push(g);
    }
  });

  if (currentMode === 'score') {
    // Score mode: render clipped GeoJSON cells colored by composite accessibility score
    var gridKey = cat;
    var grid = scoreData[gridKey] || scoreData['All'];
    grid.forEach(function(cell) {
      L.geoJSON(cell.g, {
        style: { fillColor: scoreColor(cell.s), color: 'none', weight: 0, fillOpacity: 0.55 }
      }).addTo(isoGroup);
    });
  } else {
    // Distance mode: original purple bands
    bands.forEach(function(band) {
      var key = cat + '|' + price + '|' + band;
      if (walkData[key]) {
        L.geoJSON(walkData[key], {
          style: { fillColor: colors[band], color: colors[band], weight: 0.5, fillOpacity: 0.25 }
        }).addTo(isoGroup);
      }
    });
  }

  // Add markers AFTER isochrones so they render on top
  matchedGyms.forEach(function(g) {
    var marker = L.circleMarker([g.lat, g.lon], {
      radius: 4, color: '#2c2c2a', fillColor: '#2c2c2a',
      fillOpacity: 0.7, weight: 0.5
    }).bindPopup(makePopup(g));
    marker.on('mouseover', function() { this.setRadius(8); this.setStyle({weight: 2, color: '#fff'}); });
    marker.on('mouseout', function() { if (!this.isPopupOpen()) { this.setRadius(4); this.setStyle({weight: 0.5, color: '#2c2c2a'}); } });
    marker.on('popupclose', function() { this.setRadius(4); this.setStyle({weight: 0.5, color: '#2c2c2a'}); });
    marker.addTo(markerGroup);
  });

  document.getElementById('stats').textContent = 'Showing ' + count + ' of %%TOTAL%% locations';
}

updateMap();
</script>
</body></html>"""

html_output = html_template.replace("%%CAT_OPTIONS%%", cat_options)
html_output = html_output.replace("%%PRICE_OPTIONS%%", price_options)
html_output = html_output.replace("%%TOTAL%%", str(len(gyms)))
html_output = html_output.replace("%%GYMS%%", json.dumps(gym_markers_json))
html_output = html_output.replace("%%WALK_DATA%%", json.dumps(walk_data))
html_output = html_output.replace("%%BOUNDARY%%", json.dumps(boundary_geojson))
html_output = html_output.replace("%%MBTA_LINES%%", json.dumps(mbta_lines))
html_output = html_output.replace("%%MBTA_STATIONS%%", json.dumps(mbta_stations))
html_output = html_output.replace("%%CAT_PRICE_MAP%%", json.dumps(cat_price_map))
html_output = html_output.replace("%%SCORE_DATA%%", score_data_json)

with open("boston_gym_map_v5.html", "w") as f:
    f.write(html_output)